# Train the Employee Salary Model

This notebook walks through the Random Forest salary model step by step. Run the cells from top to bottom. It saves the same model bundle used by the Flask app.

**Salary units:** the CSV does not specify currency or pay period. The model uses `CurrentSalary` as recorded, without exchange-rate conversion. Verify that the CSV values are INR and consistently monthly or annual before interpreting results as Indian-market salaries.

## 1. Import libraries and locate files

The CSV is in the project root. This path check works whether the notebook is opened from the project root or the `ML models` folder.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "EMPLOYEES_SYSTEM.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "EMPLOYEES_SYSTEM.csv"
ARTIFACT_PATH = PROJECT_DIR / "artifacts" / "salary_model.joblib"
TARGET = "CurrentSalary"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

print(f"Dataset: {DATA_PATH}")
print(f"Model output: {ARTIFACT_PATH}")

Dataset: c:\Users\kotad\OneDrive\Documents\Desktop\python C\ML\EMPLOYEES_SYSTEM.csv
Model output: c:\Users\kotad\OneDrive\Documents\Desktop\python C\ML\artifacts\salary_model.joblib


## 2. Load and inspect the data

First inspect the rows, column types, missing values, duplicate rows, and target salary range. This helps identify data issues before fitting a model.

In [2]:
raw_data = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {raw_data.shape[0]:,} rows x {raw_data.shape[1]} columns")
display(raw_data.head())
display(pd.DataFrame({"data type": raw_data.dtypes.astype(str), "missing values": raw_data.isna().sum()}))
print(f"Duplicate rows: {raw_data.duplicated().sum():,}")
display(raw_data[TARGET].describe().to_frame(name=TARGET))

Dataset shape: 5,000 rows x 11 columns


,Age,YearsExperience,YearsAtCompany,EducationLevel,Department,JobLevel,PerformanceRating,NumProjects,TrainingHours,CurrentSalary,SalaryGrowthPercent
0,50,26,22,1,2,1,3,14,63,78853.61,13.04
1,36,14,7,2,0,1,3,0,97,84564.28,11.68
2,29,6,0,1,4,1,3,9,98,44079.05,15.32
3,42,18,11,1,2,2,2,5,37,83628.45,6.08
4,40,15,13,1,2,3,4,7,37,80051.48,10.78


,data type,missing values
Age,int64,0
YearsExperience,int64,0
YearsAtCompany,int64,0
EducationLevel,int64,0
Department,int64,0
JobLevel,int64,0
PerformanceRating,int64,0
NumProjects,int64,0
TrainingHours,int64,0
CurrentSalary,float64,0


Duplicate rows: 0


,CurrentSalary
count,5000.000000
mean,88153.179420
std,24090.480288
min,32420.010000
25%,70199.725000
50%,87165.385000
75%,104754.877500
max,185774.590000


## 3. Choose model inputs

`CurrentSalary` is the value to predict. The other columns are predictors. Coded fields are handled as categories rather than as continuous measurements.

In [3]:
NUMERIC_FEATURES = [
    "Age",
    "YearsExperience",
    "YearsAtCompany",
    "NumProjects",
    "TrainingHours",
    "SalaryGrowthPercent",
]
CATEGORICAL_FEATURES = [
    "EducationLevel",
    "Department",
    "JobLevel",
    "PerformanceRating",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
REQUIRED_COLUMNS = FEATURES + [TARGET]

missing_columns = set(REQUIRED_COLUMNS) - set(raw_data.columns)
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {', '.join(sorted(missing_columns))}")

print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")

Target: CurrentSalary
Number of features: 10


## 4. Clean the rows

Remove duplicates and convert model columns to numbers. Invalid feature values become missing and are filled later by the preprocessing pipeline. Rows without a salary target cannot train a regression model, so those rows are removed.

In [4]:
original_rows = len(raw_data)
data = raw_data.drop_duplicates().copy()

for column in REQUIRED_COLUMNS:
    data[column] = pd.to_numeric(data[column], errors="coerce")

data = data.dropna(subset=[TARGET]).copy()
if len(data) < 5:
    raise ValueError("At least five usable rows are required to train and evaluate the model.")

print(f"Rows available for modeling: {len(data):,}")
print(f"Rows removed (duplicates or invalid target): {original_rows - len(data):,}")
print("Missing feature values will be imputed inside the model pipeline.")
display(data[REQUIRED_COLUMNS].isna().sum().to_frame(name="missing feature/target values"))

Rows available for modeling: 5,000
Rows removed (duplicates or invalid target): 0
Missing feature values will be imputed inside the model pipeline.


,missing feature/target values
Age,0
YearsExperience,0
YearsAtCompany,0
NumProjects,0
TrainingHours,0
SalaryGrowthPercent,0
EducationLevel,0
Department,0
JobLevel,0
PerformanceRating,0


## 5. Split into training and test data

Keep 20% aside for evaluation. The model does not see this test portion while it learns, giving us a more useful estimate of performance on unseen rows.

In [5]:
X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

Training rows: 4,000
Test rows: 1,000


## 6. Build and train the Random Forest

Preprocessing is kept inside the saved pipeline. Numeric missing values use the median. Categorical missing values use the most common value, and one-hot encoding represents category codes without implying an order.

In [6]:
preprocessing = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


## 7. Evaluate on the test rows

MAE is the average absolute error. RMSE penalizes larger errors more strongly. R-squared describes how much salary variation the model explains. MAE and RMSE use the same salary units as the CSV; no currency conversion is applied.

In [7]:
test_predictions = pipeline.predict(X_test)
metrics = {
    "mae": round(float(mean_absolute_error(y_test, test_predictions)), 2),
    "rmse": round(float(mean_squared_error(y_test, test_predictions) ** 0.5), 2),
    "r2": round(float(r2_score(y_test, test_predictions)), 4),
}

display(pd.DataFrame([metrics], index=["test set"]))

,mae,rmse,r2
test set,3967.45,5136.26,0.9535


## 8. Refit on all clean rows and save for Flask

After evaluation, fit a fresh pipeline using every cleaned row. Save its preprocessing and model together with the feature descriptions and metrics. Flask reads this bundle, so keep the dictionary keys and field information intact.

In [8]:
LABELS = {
    "Age": "Age",
    "YearsExperience": "Years of experience",
    "YearsAtCompany": "Years at company",
    "NumProjects": "Number of projects",
    "TrainingHours": "Training hours",
    "SalaryGrowthPercent": "Salary growth (%)",
    "EducationLevel": "Education level code",
    "Department": "Department code",
    "JobLevel": "Job level",
    "PerformanceRating": "Performance rating",
}

def build_field_specs(clean_data):
    field_specs = []
    for name in FEATURES:
        values = clean_data[name].dropna()
        field = {
            "name": name,
            "label": LABELS[name],
            "min": float(values.min()),
            "max": float(values.max()),
            "default": round(float(values.median()), 2),
            "step": "any",
            "categorical": name in CATEGORICAL_FEATURES,
        }
        if field["categorical"]:
            field["options"] = [
                {"value": int(value), "label": f"Code {int(value)}"}
                for value in sorted(values.unique())
            ]
            field["default"] = int(values.mode().iloc[0])
            field["step"] = "1"
        field_specs.append(field)
    return field_specs

pipeline.fit(X, y)
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
model_bundle = {
    "model": pipeline,
    "feature_columns": FEATURES,
    "fields": build_field_specs(data),
    "metrics": metrics,
    "rows_used": len(data),
    "duplicates_removed": original_rows - len(data),
}
joblib.dump(model_bundle, ARTIFACT_PATH)

print(f"Saved model bundle: {ARTIFACT_PATH}")
print(f"Rows used: {len(data):,}")
print(f"Test MAE (CSV salary units): {metrics['mae']:,.2f}")
print(f"Test RMSE (CSV salary units): {metrics['rmse']:,.2f}")
print(f"Test R-squared: {metrics['r2']:.4f}")

Saved model bundle: c:\Users\kotad\OneDrive\Documents\Desktop\python C\ML\artifacts\salary_model.joblib
Rows used: 5,000
Test MAE (CSV salary units): 3,967.45
Test RMSE (CSV salary units): 5,136.26
Test R-squared: 0.9535
